# Day 3 – Module 2: Retrieval-Augmented Generation (RAG)

**Topics covered:**
- RAG fundamentals
- Hallucination reduction through retrieval
- Naive RAG architecture
- Hybrid RAG overview
- Retrieve-and-rerank concept

**What you will do in this notebook:**
Build a complete RAG pipeline from scratch — load documents, chunk them, embed and index in a vector store, retrieve relevant context, and generate grounded answers with an LLM. Every code section maps to a numbered block in the pipeline diagram. An OpenAI API key is required for LLM calls; fallback responses are provided for review without a key.

## 1. RAG pipeline diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          RAG PIPELINE                                       │
│                                                                             │
│  ═══════════════  OFFLINE (indexing)  ═══════════════                        │
│                                                                             │
│  Documents ──▶ [Block A] Chunk ──▶ [Block B] Embed ──▶ [Block C] Index     │
│                                                         (Vector Store)      │
│                                                                             │
│  ═══════════════  ONLINE (query)  ═══════════════                           │
│                                                                             │
│  User Query                                                                 │
│       │                                                                     │
│       ▼                                                                     │
│  [Block D] Embed query                                                      │
│       │                                                                     │
│       ▼                                                                     │
│  [Block E] Retrieve top-k chunks from vector store                          │
│       │                                                                     │
│       ▼                                                                     │
│  [Block F] (Optional) Rerank retrieved chunks                               │
│       │                                                                     │
│       ▼                                                                     │
│  [Block G] Build prompt: context + question                                 │
│       │                                                                     │
│       ▼                                                                     │
│  [Block H] LLM generates answer grounded in context                        │
│       │                                                                     │
│       ▼                                                                     │
│  Answer (+ sources)                                                         │
└─────────────────────────────────────────────────────────────────────────────┘
```

Each block below is labeled with its letter from this diagram.

## 2. Setup

In [ ]:
import json, os, warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

API_KEY = os.environ.get("OPENAI_API_KEY", "")

with open("config.json") as f:
    CONFIG = json.load(f)

print(f"LLM model      : {CONFIG['llm_model']}")
print(f"Embedding model : {CONFIG['embedding_model']}")
print(f"Chunk size      : {CONFIG['chunk_size']}")
print(f"Chunk overlap   : {CONFIG['chunk_overlap']}")
print(f"Retriever top-k : {CONFIG['retriever_top_k']}")
print(f"API key set     : {bool(API_KEY)}")

## 3. RAG fundamentals

LLMs are trained on public data up to a cutoff date. They do not know your private documents, policies, or recent information. When asked about unknown topics, they may generate plausible but incorrect answers (hallucinations).

**RAG solves this**: before the LLM generates an answer, retrieve relevant documents from your own data store and include them as context in the prompt. The LLM then answers *from the provided context* rather than relying solely on its training data.

| Without RAG | With RAG |
|---|---|
| LLM answers from training data only | LLM answers from retrieved context |
| May hallucinate private/recent info | Grounded in your documents |
| No source attribution | Can cite retrieved sources |
| One-size-fits-all knowledge | Customized to your data |

## 4. Hallucination reduction through retrieval

The key mechanism: instruct the LLM to answer **only from the provided context**. If the answer is not in the context, the LLM should say so rather than invent information.

In [ ]:
# Hallucination-resistant prompt template
GROUNDED_PROMPT = """Answer the question using ONLY the provided context.
If the answer is not in the context, respond: "This information is not available in the current documents."
Do not use prior knowledge. Cite which document the answer comes from when possible.

Context:
{context}

Question: {question}

Answer:"""

print("Grounded prompt template defined.")
print(f"Variables: {{context}}, {{question}}")
print(f"Key instruction: 'Answer using ONLY the provided context'")

This prompt pattern is the foundation of hallucination reduction in RAG. The system instruction constrains the LLM to the retrieved context, and the explicit fallback ("not available") prevents fabrication.

## 5. Naive RAG — full implementation

Naive RAG is the simplest RAG architecture: retrieve top-k chunks, concatenate them as context, and pass to the LLM in a single prompt. No reranking, no multi-step reasoning.

### Document corpus

We use a set of internal IT policy documents as our corpus.

In [ ]:
# Internal IT policy documents — our knowledge base
documents = [
    {
        "id": "POL-001",
        "title": "Remote Work Policy",
        "content": (
            "Remote work is available to all full-time employees after completing 90 days of employment. "
            "Employees must use the corporate VPN (GlobalProtect) for all work activities. A dedicated workspace "
            "with reliable internet (minimum 25 Mbps) is required. Remote workers must be available during core "
            "hours (10 AM - 3 PM local time). Equipment provided: laptop, monitor, keyboard, mouse. Home office "
            "stipend of $500 is available annually for ergonomic furniture. Manager approval is required for "
            "international remote work."
        ),
    },
    {
        "id": "POL-002",
        "title": "Information Security Policy",
        "content": (
            "All employees must complete annual cybersecurity training by December 31. Passwords must be minimum "
            "12 characters with uppercase, lowercase, number, and special character. Password rotation every 90 "
            "days. Multi-factor authentication is mandatory for all cloud services. Report security incidents to "
            "security@company.com within 1 hour. Data classification: Public, Internal, Confidential, Restricted. "
            "Customer data is always Confidential or Restricted. Laptop encryption (BitLocker/FileVault) is "
            "mandatory. USB storage devices are blocked on corporate laptops."
        ),
    },
    {
        "id": "POL-003",
        "title": "Software Request Process",
        "content": (
            "All software installations require IT department approval. Submit requests through the ServiceNow "
            "catalog. Standard software (Office 365, Slack, Zoom) is pre-approved and auto-installed. Non-standard "
            "software requires manager approval and security review (1-2 business days). Open-source software must "
            "pass license compatibility check. Unapproved software installations will be flagged and removed. "
            "Software license costs above $500 require VP-level budget approval."
        ),
    },
    {
        "id": "POL-004",
        "title": "Incident Response Procedure",
        "content": (
            "Security incidents must be reported within 1 hour to security@company.com. Severity levels: Critical "
            "(data breach, ransomware), High (unauthorized access, phishing success), Medium (suspicious activity, "
            "policy violation), Low (failed login attempts, spam). Critical and High incidents trigger the Incident "
            "Response Team (IRT). IRT members: CISO, IT Security Lead, Legal, Communications. Post-incident review "
            "within 48 hours. Root cause analysis within 5 business days. All incidents logged in the SIEM system."
        ),
    },
    {
        "id": "POL-005",
        "title": "Data Retention Policy",
        "content": (
            "Email retention: 7 years for business communications, 1 year for general. Financial records: 7 years "
            "(regulatory requirement). Customer data: retained while account is active plus 2 years after closure. "
            "Employee records: 7 years after termination. Project files: 3 years after project completion. Backup "
            "retention: daily for 30 days, weekly for 90 days, monthly for 1 year. Data deletion requests processed "
            "within 30 days per GDPR requirements."
        ),
    },
    {
        "id": "POL-006",
        "title": "Acceptable Use Policy",
        "content": (
            "Corporate devices and network are for business use. Limited personal use is acceptable during breaks. "
            "Prohibited: accessing illegal content, cryptocurrency mining, unauthorized file sharing, bypassing "
            "security controls, connecting unauthorized devices. Social media use must comply with the social media "
            "policy. Personal cloud storage (Dropbox, Google Drive personal) must not store corporate data. "
            "Violations may result in disciplinary action up to termination."
        ),
    },
    {
        "id": "POL-007",
        "title": "Travel and Expense Policy",
        "content": (
            "Business travel requires manager pre-approval. Economy class for flights under 6 hours, business class "
            "for longer flights. Hotel maximum: $200/night domestic, $300/night international. Meal per diem: $75 "
            "domestic, $100 international. Receipts required for expenses over $25. Expense reports due within 10 "
            "business days of trip completion. Corporate credit card must be used when available. Mileage "
            "reimbursement: $0.67 per mile."
        ),
    },
    {
        "id": "POL-008",
        "title": "Employee Onboarding",
        "content": (
            "New employee orientation on first day: HR paperwork, IT setup, office tour, team introduction. IT "
            "provisions: laptop, email, Slack, VPN access within 24 hours. Required training in first 30 days: "
            "cybersecurity awareness, code of conduct, anti-harassment, data privacy. Badge access provisioned on "
            "day one. Buddy system: each new hire paired with a tenured team member for 90 days. Probation period: "
            "90 days with mid-point review at 45 days and final review at 90 days."
        ),
    },
]

print(f"Corpus: {len(documents)} documents")
for doc in documents:
    print(f"  [{doc['id']}] {doc['title']} ({len(doc['content'])} chars)")

### Block A: Chunk documents

Large documents must be split into smaller chunks so the embedding model and LLM can process them effectively. `RecursiveCharacterTextSplitter` splits by paragraphs, then sentences, then words, respecting the chunk size limit.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Convert to LangChain Documents and split
all_chunks = []
for doc in documents:
    lc_doc = Document(
        page_content=doc["content"],
        metadata={"id": doc["id"], "title": doc["title"]},
    )
    chunks = splitter.split_documents([lc_doc])
    all_chunks.extend(chunks)

print(f"[Block A] {len(documents)} documents → {len(all_chunks)} chunks")
print(f"Chunk size range: {min(len(c.page_content) for c in all_chunks)} – {max(len(c.page_content) for c in all_chunks)} chars")
print(f"\nSample chunk (#{1}):")
print(f"  Source: {all_chunks[0].metadata['title']}")
print(f"  Text: {all_chunks[0].page_content[:150]}...")

### Block B + C: Embed chunks and build vector index

Embed all chunks using `sentence-transformers` and store them in a Chroma vector database.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=CONFIG["embedding_model"])

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="rag_demo",
)

print(f"[Block B+C] Embedded and indexed {vectorstore._collection.count()} chunks")
print(f"Embedding model: {CONFIG['embedding_model']}")

### Block D + E: Embed query and retrieve top-k chunks

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": CONFIG["retriever_top_k"]})

query = "What are the password requirements?"
retrieved = retriever.invoke(query)

print(f"[Block D+E] Query: '{query}'")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"  {i}. [{doc.metadata.get('id', '?')}] {doc.metadata.get('title', '?')}")
    print(f"     {doc.page_content[:120]}...")
    print()

### Block G + H: Build prompt with context and generate answer

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=CONFIG["llm_model"],
    temperature=CONFIG["temperature"],
    api_key=API_KEY or "sk-placeholder",
)
parser = StrOutputParser()

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", GROUNDED_PROMPT),
])

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("id", "unknown")
        formatted.append(f"[{source}] {doc.page_content}")
    return "\n\n".join(formatted)

context_text = format_docs(retrieved)

if API_KEY:
    answer = (rag_prompt | llm | parser).invoke({
        "context": context_text,
        "question": query,
    })
else:
    answer = (
        "[Simulated] Passwords must be minimum 12 characters with uppercase, lowercase, "
        "number, and special character. Password rotation is required every 90 days. "
        "(Source: POL-002 Information Security Policy)"
    )

print(f"[Block G+H] Question: {query}")
print(f"Answer: {answer}")

### Complete naive RAG chain

Combining all blocks into a single LangChain chain using LCEL.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

print("Naive RAG chain assembled: retriever | format_docs | prompt | llm | parser")

In [ ]:
test_questions = [
    "How long are financial records retained?",
    "What happens if I install unapproved software?",
    "What is the meal per diem for international travel?",
    "What training is required in the first 30 days of employment?",
]

simulated_answers = [
    "Financial records are retained for 7 years as a regulatory requirement. (Source: POL-005)",
    "Unapproved software installations will be flagged and removed. (Source: POL-003)",
    "International meal per diem is $100 per day. (Source: POL-007)",
    "Required training: cybersecurity awareness, code of conduct, anti-harassment, data privacy. (Source: POL-008)",
]

for i, q in enumerate(test_questions):
    if API_KEY:
        answer = rag_chain.invoke(q)
    else:
        answer = simulated_answers[i]
    print(f"Q: {q}")
    print(f"A: {answer}\n")

### Naive RAG with source attribution

In [ ]:
from langchain_core.runnables import RunnableLambda

def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return {
        "context": format_docs(docs),
        "question": question,
        "source_ids": [d.metadata.get("id", "?") for d in docs],
        "source_titles": [d.metadata.get("title", "?") for d in docs],
    }

def generate_answer(data):
    if API_KEY:
        answer = (rag_prompt | llm | parser).invoke({
            "context": data["context"],
            "question": data["question"],
        })
    else:
        answer = "[Simulated] Answer grounded in retrieved documents."
    return {**data, "answer": answer}

rag_with_sources = RunnableLambda(retrieve_and_format) | RunnableLambda(generate_answer)

result = rag_with_sources.invoke("What is the severity level for a data breach?")
print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Sources: {', '.join(f'{sid} ({stitle})' for sid, stitle in zip(result['source_ids'], result['source_titles']))}")

### Handling out-of-scope questions

In [ ]:
out_of_scope = "What is the company's stock price?"

result = rag_with_sources.invoke(out_of_scope)
print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Sources retrieved: {result['source_ids']}")
print()
print("The grounded prompt instructs the LLM to say the information is not available,")
print("rather than hallucinating a stock price.")

## 6. Chunking strategies

Chunk size and overlap affect retrieval quality. Smaller chunks are more precise but may lose context; larger chunks retain context but dilute relevance.

In [ ]:
# Compare different chunk sizes
for chunk_size in [200, 500, 1000]:
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=50,
    )
    test_doc = Document(page_content=documents[0]["content"])
    test_chunks = test_splitter.split_documents([test_doc])
    print(f"Chunk size {chunk_size}: {len(test_chunks)} chunks, avg {sum(len(c.page_content) for c in test_chunks)//len(test_chunks)} chars")

| Chunk size | Pros | Cons | Best for |
|---|---|---|---|
| Small (200) | Precise retrieval | May lose context across sentences | FAQ, short answers |
| Medium (500) | Good balance | Default choice | General RAG |
| Large (1000) | Retains full context | May dilute relevance | Long-form answers |

## 7. Hybrid RAG overview

Naive RAG uses only vector (semantic) search. **Hybrid RAG** combines semantic search with keyword search (e.g., BM25) to handle cases where exact terms matter.

| Search type | Finds | Misses |
|---|---|---|
| Semantic (vector) | Conceptually similar text | Exact IDs, codes, names |
| Keyword (BM25) | Exact term matches | Synonyms, paraphrases |
| **Hybrid** | Both semantic and exact matches | — |

Hybrid RAG merges results from both retrievers using **reciprocal rank fusion** (RRF) or simple interleaving.

In [ ]:
# Simulated hybrid retrieval demonstration
# In production: use langchain.retrievers.EnsembleRetriever with BM25Retriever + vector retriever

from collections import Counter

query = "POL-002 password rotation"

# Vector retrieval (semantic)
vector_results = retriever.invoke(query)
vector_ids = [d.metadata.get("id") for d in vector_results]

# Simulated BM25 retrieval (keyword) — matches exact "POL-002" and "password"
bm25_ids = ["POL-002", "POL-008", "POL-001"]  # simulated keyword matches

# Reciprocal Rank Fusion
def reciprocal_rank_fusion(ranked_lists, k=60):
    scores = Counter()
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] += 1.0 / (k + rank)
    return scores.most_common()

fused = reciprocal_rank_fusion([vector_ids, bm25_ids])

print(f"Query: '{query}'")
print(f"Vector results : {vector_ids}")
print(f"BM25 results   : {bm25_ids}")
print(f"Fused ranking  : {[doc_id for doc_id, _ in fused]}")
print(f"\nPOL-002 appears in both lists → ranked highest after fusion.")

In production, use `EnsembleRetriever` from LangChain:

```python
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(chunks, k=3)
vector = vectorstore.as_retriever(search_kwargs={"k": 3})

hybrid = EnsembleRetriever(
    retrievers=[bm25, vector],
    weights=[0.4, 0.6],  # weight semantic higher
)
```

Hybrid RAG is most valuable when users search by exact identifiers (policy numbers, product codes) alongside natural language questions.

## 8. Retrieve-and-rerank concept

Standard retrieval returns the top-k most *similar* chunks based on embedding distance. But embedding similarity is not always the best measure of *relevance* to the specific question.

**Reranking**: retrieve a larger candidate set (e.g., top-20), then score each candidate against the query using a cross-encoder model, and keep only the top-3.

```
Retrieve top-20 (fast, approximate) → Rerank with cross-encoder → Return top-3 (precise)
```

In [ ]:
from sentence_transformers import CrossEncoder

# Cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "What is the process for requesting new software?"

# Step 1: retrieve more candidates (top-6)
candidates = vectorstore.as_retriever(search_kwargs={"k": 6}).invoke(query)

# Step 2: score each candidate against the query
pairs = [(query, doc.page_content) for doc in candidates]
scores = reranker.predict(pairs)

# Step 3: rerank by score
ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

print(f"Query: '{query}'")
print(f"\nBefore reranking (vector similarity order):")
for i, doc in enumerate(candidates, 1):
    print(f"  {i}. [{doc.metadata.get('id')}] {doc.page_content[:80]}...")

print(f"\nAfter reranking (cross-encoder relevance):")
for i, (doc, score) in enumerate(ranked[:3], 1):
    print(f"  {i}. [{doc.metadata.get('id')}] score={score:.4f} | {doc.page_content[:80]}...")

The cross-encoder evaluates the query and document *together* (not independently), producing a more accurate relevance score. This is slower than embedding similarity but much more precise for the final ranking.

| Approach | Speed | Precision | When to use |
|---|---|---|---|
| Vector only (top-k) | Fast | Good | Small/medium corpora, low-latency needs |
| Retrieve + rerank | Slower | High | Large corpora, high-precision needs |

## 9. End-to-end RAG walkthrough

This section implements the complete RAG pipeline in numbered blocks, matching the diagram from Section 1.

### Code-to-architecture mapping

| Block | Diagram label | Implementation |
|---|---|---|
| A | Chunk documents | `RecursiveCharacterTextSplitter` |
| B | Embed chunks | `HuggingFaceEmbeddings.embed_documents()` |
| C | Build vector index | `Chroma.from_documents()` |
| D | Embed query | Handled by retriever internally |
| E | Retrieve top-k | `retriever.invoke(query)` |
| F | Rerank (optional) | `CrossEncoder.predict()` |
| G | Build prompt | `ChatPromptTemplate` with context + question |
| H | LLM generates answer | `ChatOpenAI.invoke()` via chain |

In [ ]:
# ── Block A: Chunk ──
print("=" * 60)
print("BLOCK A: Chunking documents")
print("=" * 60)

e2e_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
)

e2e_docs = []
for doc in documents:
    lc_doc = Document(
        page_content=doc["content"],
        metadata={"id": doc["id"], "title": doc["title"]},
    )
    e2e_docs.extend(e2e_splitter.split_documents([lc_doc]))

print(f"  {len(documents)} documents → {len(e2e_docs)} chunks")

In [ ]:
# ── Block B + C: Embed and index ──
print("=" * 60)
print("BLOCK B+C: Embedding and indexing")
print("=" * 60)

e2e_store = Chroma.from_documents(
    documents=e2e_docs,
    embedding=embeddings,
    collection_name="e2e_rag",
)
print(f"  Indexed {e2e_store._collection.count()} chunks in Chroma")

In [ ]:
# ── Block D + E: Query and retrieve ──
print("=" * 60)
print("BLOCK D+E: Query embedding and retrieval")
print("=" * 60)

e2e_retriever = e2e_store.as_retriever(search_kwargs={"k": 3})
e2e_query = "What training must new employees complete in their first month?"
e2e_retrieved = e2e_retriever.invoke(e2e_query)

print(f"  Query: '{e2e_query}'")
print(f"  Retrieved {len(e2e_retrieved)} chunks:")
for i, doc in enumerate(e2e_retrieved, 1):
    print(f"    {i}. [{doc.metadata.get('id')}] {doc.page_content[:100]}...")

In [ ]:
# ── Block G + H: Prompt and generate ──
print("=" * 60)
print("BLOCK G+H: Prompt construction and LLM generation")
print("=" * 60)

e2e_context = format_docs(e2e_retrieved)

if API_KEY:
    e2e_answer = (rag_prompt | llm | parser).invoke({
        "context": e2e_context,
        "question": e2e_query,
    })
else:
    e2e_answer = (
        "[Simulated] New employees must complete cybersecurity awareness, code of conduct, "
        "anti-harassment, and data privacy training within the first 30 days. (Source: POL-008)"
    )

print(f"  Question: {e2e_query}")
print(f"  Answer: {e2e_answer}")
print(f"  Sources: {[d.metadata.get('id') for d in e2e_retrieved]}")

### Evaluating retrieval and answer quality

In [ ]:
# Ground truth: which policy document should answer each question?
ground_truth = [
    ("What are the password requirements?", "POL-002"),
    ("Can I work from another country?", "POL-001"),
    ("How do I request Photoshop?", "POL-003"),
    ("What is a Critical severity incident?", "POL-004"),
    ("How long are emails retained?", "POL-005"),
    ("Can I mine cryptocurrency on my work laptop?", "POL-006"),
    ("What is the hotel limit for domestic travel?", "POL-007"),
    ("Who is assigned as my buddy during onboarding?", "POL-008"),
]

hits = 0
for question, expected_id in ground_truth:
    docs = e2e_retriever.invoke(question)
    retrieved_ids = [d.metadata.get("id") for d in docs]
    found = expected_id in retrieved_ids
    hits += found
    status = "HIT" if found else "MISS"
    print(f"  [{status}] Q: {question[:50]:50s} Expected: {expected_id} | Got: {retrieved_ids}")

recall = hits / len(ground_truth)
print(f"\nRetrieval Recall@{CONFIG['retriever_top_k']}: {recall:.0%} ({hits}/{len(ground_truth)})")

### Production-ready RAG function

In [ ]:
import time

def rag_query(question, retriever, llm, prompt_template, api_key, top_k=3):
    start = time.perf_counter()

    # Retrieve
    docs = retriever.invoke(question)
    context = format_docs(docs)
    sources = [{"id": d.metadata.get("id"), "title": d.metadata.get("title")} for d in docs]

    # Generate
    if api_key:
        answer = (prompt_template | llm | parser).invoke({
            "context": context,
            "question": question,
        })
    else:
        answer = "[Simulated] Grounded answer based on retrieved context."

    elapsed = time.perf_counter() - start

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "num_chunks_retrieved": len(docs),
        "latency_seconds": round(elapsed, 3),
    }

# Test
result = rag_query(
    "What is the probation period for new employees?",
    e2e_retriever, llm, rag_prompt, API_KEY,
)
for k, v in result.items():
    print(f"  {k}: {v}")

### Batch RAG processing

In [ ]:
batch_questions = [
    "What equipment is provided for remote workers?",
    "How quickly must security incidents be reported?",
    "What is the mileage reimbursement rate?",
    "What happens during the 45-day mid-point review?",
]

print(f"Processing {len(batch_questions)} questions...\n")
for q in batch_questions:
    result = rag_query(q, e2e_retriever, llm, rag_prompt, API_KEY)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer'][:150]}...")
    print(f"   Sources: {[s['id'] for s in result['sources']]}  ({result['latency_seconds']}s)")
    print()

## 10. Frameworks and components reference

| Component | Purpose | LangChain class |
|---|---|---|
| Text splitter | Chunk documents | `RecursiveCharacterTextSplitter` |
| Embeddings | Convert text → vectors | `HuggingFaceEmbeddings` |
| Vector store | Index and search vectors | `Chroma`, `FAISS` |
| Retriever | Fetch top-k relevant chunks | `VectorStoreRetriever` |
| Reranker | Re-score candidates | `CrossEncoder` (sentence-transformers) |
| Prompt | Structure context + question | `ChatPromptTemplate` |
| LLM | Generate grounded answer | `ChatOpenAI` |
| Output parser | Extract text from response | `StrOutputParser` |

Refer to Module 1 for LangChain architecture details.

## Summary

- **RAG** = retrieve relevant documents, then generate answers grounded in context
- **Hallucination reduction**: instruct the LLM to answer only from provided context
- **Naive RAG**: chunk → embed → index → retrieve → prompt → LLM → answer
- **Hybrid RAG**: combine semantic (vector) search with keyword (BM25) search
- **Retrieve-and-rerank**: over-retrieve candidates, then rerank with a cross-encoder for higher precision
- **Source attribution**: return retrieved document IDs alongside answers for transparency

Next: **Module 3 — AI Evaluation & Responsible AI** covers how to evaluate RAG outputs and apply responsible AI principles before deployment.

## Cleanup

In [ ]:
try:
    vectorstore._client.delete_collection("rag_demo")
    e2e_store._client.delete_collection("e2e_rag")
    print("Temporary Chroma collections removed.")
except Exception:
    pass

print("Notebook complete.")